# Module 2: Building the SciPy Knowledge Base

**Duration:** 1.5-2 hours

## Learning Objectives

By the end of this module, you will:
- Scrape and process documentation for RAG
- Implement effective chunking strategies
- Build a production-ready vector store with ChromaDB
- Test retrieval quality

---

## Setup

First, let's import our dependencies and set up paths.

### Model roster note (so this notebook stays current)

Model names and recommended defaults can change over time. To avoid the notebook becoming outdated:

- Keep model choices in one place (see `EMBEDDING_MODEL` above).
- When refreshing, check the official OpenAI **Models** list and update the constants rather than editing many cells.

In [1]:
import os
import sys
import json
import datetime
from pathlib import Path

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from dotenv import load_dotenv
load_dotenv()

# Verify we have what we need
print(f"OpenAI API Key: {'Set' if os.getenv('OPENAI_API_KEY') else 'Not set'}")

# RAG config (keep these as constants so you can refresh model choices easily)
EMBEDDING_MODEL = "text-embedding-3-small"

# Prefer calling embeddings via the OpenAI SDK (future-proof + better error surfacing).
# If you prefer Chroma to call OpenAI directly, set this to False.
USE_OPENAI_SDK_EMBEDDINGS = True

try:
    from openai import OpenAI
    openai_client = OpenAI()
except Exception as e:
    USE_OPENAI_SDK_EMBEDDINGS = False
    openai_client = None
    print("⚠️ OpenAI SDK not available; falling back to Chroma embedding function.")


OpenAI API Key: Set


---

## 1. Web Scraping SciPy Documentation

### Scraping Strategy

When scraping documentation, we need to be respectful and efficient:

1. **Respect robots.txt**: Check what's allowed to scrape
2. **Rate limiting**: Don't overwhelm the server (we use 0.5s delay)
3. **User-Agent**: Identify ourselves clearly
4. **Caching**: Save scraped data to avoid re-scraping

Let's look at our scraper implementation.

In [2]:
# View the scraper code
from scraper import SciPyDocsScraper, create_sample_dataset, ScrapedDocument

# Show available modules
print("SciPy modules we can scrape:")
for module in SciPyDocsScraper.MODULES:
    print(f"  - scipy.{module}")

SciPy modules we can scrape:
  - scipy.cluster
  - scipy.constants
  - scipy.fft
  - scipy.integrate
  - scipy.interpolate
  - scipy.io
  - scipy.linalg
  - scipy.ndimage
  - scipy.optimize
  - scipy.signal
  - scipy.sparse
  - scipy.spatial
  - scipy.special
  - scipy.stats


### Option A: Use Sample Dataset (Fast, No Network)

For this workshop, we'll use a pre-built sample dataset to save time. This is recommended for learning.

### Data freshness and provenance (recommended)

Vector search quality depends heavily on **what docs you scraped** and **when** you scraped them.

To keep this notebook from going stale, we attach provenance fields to each document (and propagate them into chunk metadata):

- `source_url`: where the text came from (prefer versioned docs URLs when possible)
- `retrieved_at`: ISO timestamp of when you built the dataset/KB
- `scipy_doc_version`: optional (if you know the doc version you scraped)

If you use **Option A (sample dataset)**, treat it as a *demo corpus*; for a production-ish KB, use **Option B (live scrape)** and store URLs + timestamps.

In [3]:
# Create sample dataset
sample_docs = create_sample_dataset()

print(f"Sample dataset contains {len(sample_docs)} documents")
print("\nSample document:")
print(f"  Title: {sample_docs[0]['title']}")
print(f"  Module: {sample_docs[0]['module']}")
print(f"  Signature: {sample_docs[0]['signature'][:80]}...")

Sample dataset contains 5 documents

Sample document:
  Title: scipy.optimize.minimize
  Module: scipy.optimize
  Signature: scipy.optimize.minimize(fun, x0, args=(), method=None, jac=None, hess=None, hess...


In [4]:
# Save sample dataset
data_dir = Path.cwd().parent / 'data' / 'processed'
data_dir.mkdir(parents=True, exist_ok=True)

with open(data_dir / 'scipy_sample.json', 'w') as f:
    json.dump(sample_docs, f, indent=2)
    
print(f"Saved to {data_dir / 'scipy_sample.json'}")

Saved to /Users/cynthia/Desktop/scipy_rag/data/processed/scipy_sample.json


### Option B: Live Scraping (Slower, Complete Data)

If you want the full experience, uncomment and run this cell to scrape real documentation. **Warning: This takes several minutes.**

Freshness tips:
- Prefer scraping **versioned** docs URLs (so the KB can be reproduced later).
- Persist `source_url`, `retrieved_at`, and (if known) `scipy_doc_version` into your dataset so you can refresh intentionally.

In [5]:
# UNCOMMENT TO SCRAPE LIVE DATA (may take several minutes depending on modules)

scraper = SciPyDocsScraper(
    delay=0.5,  # Be polite wait 0.5s between requests
    output_dir=str(Path.cwd().parent / "data" / "raw")
)

# Scrape a subset of modules (faster for demo)
modules_to_scrape = ["optimize", "integrate"]  # Add more as needed
live_docs = scraper.scrape_all(modules=modules_to_scrape)

# Optional: stamp provenance on the scraped docs (highly recommended)
# now_iso = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
# for d in live_docs:
#     d.setdefault("retrieved_at", now_iso)
#     d.setdefault("scipy_doc_version", None)  # set if you know it (e.g., "1.17.0")
#     d.setdefault("source_url", d.get("url") or d.get("source_url"))

print(f"\nScraped {len(live_docs)} documents from live site")


Scraping scipy.optimize...
Found 4 function pages


scipy.optimize: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.74it/s]


Saved 4 documents to /Users/cynthia/Desktop/scipy_rag/data/raw/scipy_optimize.json

Scraping scipy.integrate...
Found 1 function pages


scipy.integrate: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.72it/s]

Saved 1 documents to /Users/cynthia/Desktop/scipy_rag/data/raw/scipy_integrate.json
Saved 5 documents to /Users/cynthia/Desktop/scipy_rag/data/raw/scipy_all.json

Total documents scraped: 5

Scraped 5 documents from live site


### Examining the Scraped Data

Let's look at what information we extracted from each documentation page.

In [6]:
# Load our working dataset
with open(data_dir / 'scipy_sample.json', 'r') as f:
    documents = json.load(f)

# Examine one document in detail
doc = documents[0]  # scipy.optimize.minimize

print("=" * 60)
print(f"Function: {doc['title']}")
print("=" * 60)

print(f"\n📦 Module: {doc['module']}")
print(f"📝 Type: {doc['doc_type']}")

print(f"\n📋 Signature:\n{doc['signature']}")

print(f"\n📖 Description:\n{doc['description']}")

print(f"\n🔧 Parameters (first 500 chars):\n{doc['parameters'][:500]}...")

print(f"\n📤 Returns:\n{doc['returns']}")

print(f"\n💻 Examples (first 500 chars):\n{doc['examples'][:500]}...")

Function: scipy.optimize.minimize

📦 Module: scipy.optimize
📝 Type: function

📋 Signature:
scipy.optimize.minimize(fun, x0, args=(), method=None, jac=None, hess=None, hessp=None, bounds=None, constraints=(), tol=None, callback=None, options=None)

📖 Description:
Minimization of scalar function of one or more variables.

🔧 Parameters (first 500 chars):
fun : callable
    The objective function to be minimized.
x0 : ndarray, shape (n,)
    Initial guess.
method : str or callable, optional
    Type of solver.
bounds : sequence or Bounds, optional
    Bounds on variables....

📤 Returns:
res : OptimizeResult
    The optimization result as an OptimizeResult object.

💻 Examples (first 500 chars):
>>> from scipy.optimize import minimize
>>> from scipy.optimize import rosen
>>> x0 = [1.3, 0.7, 0.8, 1.9, 1.2]
>>> res = minimize(rosen, x0, method="Nelder-Mead", tol=1e-6)
>>> res.x
array([1., 1., 1., 1., 1.])...


In [8]:
# Attach provenance to documents (keeps the KB auditable + easier to refresh)
now_iso = datetime.datetime.now(datetime.UTC).replace(microsecond=0).isoformat() + "Z"

for d in documents:
    d.setdefault("retrieved_at", now_iso)
    d.setdefault("source_url", None)
    d.setdefault("scipy_doc_version", None)

print("Provenance fields ensured on all documents.")

Provenance fields ensured on all documents.


---

## 2. Document Chunking Strategies

### Why Chunking Matters

Chunking is **critical** for RAG quality. Poor chunking leads to:
- Retrieved context missing key information
- Code examples split in the middle (useless!)
- Related information separated across chunks

### Chunking Strategies

| Strategy | Pros | Cons | Best For |
|----------|------|------|----------|
| **Fixed-size** | Simple, predictable | May split sentences/code | Uniform text |
| **Recursive** | Respects boundaries | May create uneven chunks | Prose/articles |
| **Code-aware** | Preserves code blocks | More complex | Documentation |
| **Semantic** | Coherent topics | Expensive (needs embeddings) | Long documents |

In [9]:
from chunker import (
    fixed_size_chunker,
    recursive_text_splitter,
    code_aware_chunker,
    chunk_scipy_document,
    chunk_documents,
    Chunk,
    TokenCounter
)

### 2.1 Fixed-Size Chunking

In [10]:
# Sample text with code
sample_text = documents[1]['full_text']  # curve_fit

# Fixed-size chunking
fixed_chunks = fixed_size_chunker(sample_text, chunk_size=300, overlap=50)

print(f"Fixed-size chunking produced {len(fixed_chunks)} chunks")
print("\n" + "="*60)

for i, chunk in enumerate(fixed_chunks[:3]):  # Show first 3
    print(f"\nChunk {i} ({len(chunk)} chars):")
    print("-" * 40)
    print(chunk[:200] + "..." if len(chunk) > 200 else chunk)

Fixed-size chunking produced 1 chunks


Chunk 0 (84 chars):
----------------------------------------
scipy.optimize.curve_fit - Use non-linear least squares to fit a function to data...


**Problem**: Notice how fixed-size chunking might cut text mid-sentence or mid-code-block!

### 2.2 Recursive Text Splitting

In [11]:
# Recursive chunking - respects paragraph/sentence boundaries
recursive_chunks = recursive_text_splitter(sample_text, chunk_size=300, overlap=50)

print(f"Recursive chunking produced {len(recursive_chunks)} chunks")
print("\n" + "="*60)

for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"\nChunk {i} ({len(chunk)} chars):")
    print("-" * 40)
    print(chunk[:200] + "..." if len(chunk) > 200 else chunk)

Recursive chunking produced 1 chunks


Chunk 0 (84 chars):
----------------------------------------
scipy.optimize.curve_fit - Use non-linear least squares to fit a function to data...


**Better**: Recursive splitting tries to keep sentences together!

### 2.3 Code-Aware Chunking

In [12]:
# Code-aware chunking - keeps code blocks intact
code_chunks = code_aware_chunker(sample_text, chunk_size=400, overlap=50)

print(f"Code-aware chunking produced {len(code_chunks)} chunks")
print("\n" + "="*60)

for i, chunk in enumerate(code_chunks[:3]):
    print(f"\nChunk {i} ({len(chunk)} chars):")
    print("-" * 40)
    # Check if chunk contains code
    has_code = '>>>' in chunk or 'import' in chunk or 'def ' in chunk
    print(f"[Contains code: {has_code}]")
    print(chunk[:250] + "..." if len(chunk) > 250 else chunk)

Code-aware chunking produced 1 chunks


Chunk 0 (84 chars):
----------------------------------------
[Contains code: False]
scipy.optimize.curve_fit - Use non-linear least squares to fit a function to data...


**Best for docs**: Code-aware chunking prioritizes keeping code examples complete!

### 2.4 Structured Document Chunking

For SciPy docs, we use a special strategy that creates multiple chunk types:
1. **Summary chunk**: Signature + description + key parameters
2. **Examples chunk**: Code examples (kept together)
3. **Full-text chunks**: Remaining content

In [13]:
# Chunk a single document with our specialized function
doc_chunks = chunk_scipy_document(documents[0], strategy="code_aware", chunk_size=600)

print(f"Document '{documents[0]['title']}' produced {len(doc_chunks)} chunks:")
print("\n" + "="*60)

for chunk in doc_chunks:
    print(f"\n📄 {chunk.chunk_id}")
    print(f"   Type: {chunk.metadata['chunk_type']}")
    print(f"   Size: {len(chunk.text)} chars")
    print(f"   Preview: {chunk.text[:100]}...")

Document 'scipy.optimize.minimize' produced 2 chunks:


📄 minimize_summary
   Type: summary
   Size: 448 chars
   Preview: scipy.optimize.minimize(fun, x0, args=(), method=None, jac=None, hess=None, hessp=None, bounds=None,...

📄 minimize_examples
   Type: examples
   Size: 235 chars
   Preview: Examples for minimize:

>>> from scipy.optimize import minimize
>>> from scipy.optimize import rosen...


### 2.5 Chunking All Documents

In [14]:
# Chunk all documents
all_chunks = chunk_documents(documents, strategy="code_aware", chunk_size=600, overlap=100)

print(f"Total chunks created: {len(all_chunks)}")

# Statistics
chunk_types = {}
chunk_sizes = [len(c.text) for c in all_chunks]

for chunk in all_chunks:
    ct = chunk.metadata['chunk_type']
    chunk_types[ct] = chunk_types.get(ct, 0) + 1

print(f"\nChunk type distribution:")
for ct, count in chunk_types.items():
    print(f"  {ct}: {count}")

print(f"\nChunk size statistics:")
print(f"  Min: {min(chunk_sizes)} chars")
print(f"  Max: {max(chunk_sizes)} chars")
print(f"  Avg: {sum(chunk_sizes) / len(chunk_sizes):.0f} chars")

Total chunks created: 10

Chunk type distribution:
  summary: 5
  examples: 5

Chunk size statistics:
  Min: 147 chars
  Max: 488 chars
  Avg: 310 chars


In [15]:
# Propagate document provenance into chunk metadata (helps audits + refreshes)
by_title = {d.get("title"): d for d in documents}

for ch in all_chunks:
    fn = ch.metadata.get("function_name")
    doc = by_title.get(fn)
    if doc is None:
        continue

    ch.metadata.setdefault("source_url", doc.get("source_url"))
    ch.metadata.setdefault("retrieved_at", doc.get("retrieved_at"))
    ch.metadata.setdefault("scipy_doc_version", doc.get("scipy_doc_version"))

print("Chunk metadata updated with provenance where available.")

Chunk metadata updated with provenance where available.


---

## 3. Building the Vector Store

Now let's embed our chunks and store them in ChromaDB.

In [16]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# Create a persistent client
chroma_path = Path.cwd().parent / "chroma_db"
chroma_path.mkdir(exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(chroma_path))
print(f"ChromaDB initialized at {chroma_path}")

ChromaDB initialized at /Users/cynthia/Desktop/scipy_rag/chroma_db


In [17]:
# Create / reset the collection (safe to re-run)
try:
    chroma_client.delete_collection("scipy_docs")
    print("Deleted existing collection")
except Exception:
    pass

collection_kwargs = {
    "name": "scipy_docs",
    "metadata": {
        "description": "SciPy documentation for RAG",
        "hnsw:space": "cosine"  # Use cosine similarity
    }
}

if USE_OPENAI_SDK_EMBEDDINGS:
    # We'll supply embeddings ourselves via the OpenAI SDK.
    collection = chroma_client.create_collection(**collection_kwargs)
else:
    # Let Chroma call OpenAI for embeddings (convenient, but less flexible).
    openai_ef = OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name=EMBEDDING_MODEL
    )
    collection = chroma_client.create_collection(
        **collection_kwargs,
        embedding_function=openai_ef
    )

print(f"Created collection: {collection.name}")
print(f"Embedding mode: {'OpenAI SDK' if USE_OPENAI_SDK_EMBEDDINGS else 'Chroma embedding function'}")
print(f"Embedding model: {EMBEDDING_MODEL}")

Created collection: scipy_docs
Embedding mode: OpenAI SDK
Embedding model: text-embedding-3-small


In [18]:
def embed_texts(texts):
    # Embed a list of strings using the configured embedding model.
    # Returns: List[List[float]] embeddings
    if not USE_OPENAI_SDK_EMBEDDINGS:
        raise RuntimeError("embed_texts called but USE_OPENAI_SDK_EMBEDDINGS is False")

    if openai_client is None:
        raise RuntimeError("OpenAI client not initialized")

    resp = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [item.embedding for item in resp.data]

In [19]:
from tqdm import tqdm

# Add chunks to collection in batches
batch_size = 50

for i in tqdm(range(0, len(all_chunks), batch_size), desc="Indexing chunks"):
    batch = all_chunks[i:i + batch_size]

    ids = [chunk.chunk_id for chunk in batch]
    docs = [chunk.text for chunk in batch]
    metas = [chunk.metadata for chunk in batch]

    if USE_OPENAI_SDK_EMBEDDINGS:
        embs = embed_texts(docs)
        collection.add(
            ids=ids,
            documents=docs,
            metadatas=metas,
            embeddings=embs
        )
    else:
        collection.add(
            ids=ids,
            documents=docs,
            metadatas=metas
        )

print(f"\n✅ Added {collection.count()} chunks to the vector store")

Indexing chunks:   0%|                                                                                                                                                      | 0/1 [00:01<?, ?it/s]


TypeError: argument 'metadatas': Cannot convert Python object to MetadataValue

---

## 4. Retrieval Testing

Let's test our knowledge base with various queries!

In [ ]:
def search_scipy(query: str, n_results: int = 3, filter_module: str = None):
    """Search the SciPy knowledge base."""
    where = {"module": filter_module} if filter_module else None

    if USE_OPENAI_SDK_EMBEDDINGS:
        query_emb = embed_texts([query])[0]
        results = collection.query(
            query_embeddings=[query_emb],
            n_results=n_results,
            where=where
        )
    else:
        results = collection.query(
            query_texts=[query],
            n_results=n_results,
            where=where
        )

    print(f"🔍 Query: '{query}'")
    if filter_module:
        print(f"   Filter: {filter_module}")
    print("=" * 60)

    for i, (doc, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
    ):
        fn = metadata.get("function_name", "unknown")
        ct = metadata.get("chunk_type", "chunk")
        mod = metadata.get("module", "unknown")
        print(f"\n[{i + 1}] {fn} ({ct})")
        print(f"    Module: {mod}")
        print(f"    Distance: {distance:.4f}")
        if metadata.get("source_url"):
            print(f"    Source: {metadata['source_url']}")
        if metadata.get("retrieved_at"):
            print(f"    Retrieved: {metadata['retrieved_at']}")
        print(f"    Preview: {doc[:150]}...")

    return results

In [ ]:
# Test queries
search_scipy("How do I fit a curve to my experimental data?")

In [ ]:
search_scipy("calculate the integral of a function")

In [ ]:
search_scipy("solve a system of linear equations")

In [ ]:
search_scipy("filter a signal with butterworth")

In [ ]:
# Test with module filter
search_scipy("optimization", n_results=2, filter_module="scipy.optimize")

### Debugging Poor Retrieval Results

If retrieval quality is poor, common issues include:

1. **Chunks too small**: Important context split across chunks
2. **Chunks too large**: Diluted with irrelevant content
3. **Code not preserved**: Examples split mid-block
4. **Missing metadata**: Can't filter effectively

In [ ]:
# Check what chunks contain for a specific function
def inspect_function_chunks(function_name: str):
    """See all chunks for a specific function."""
    results = collection.get(
        where={"function_name": function_name}
    )
    
    print(f"Chunks for '{function_name}':")
    print("=" * 60)
    
    for i, (doc_id, doc, metadata) in enumerate(zip(
        results['ids'],
        results['documents'],
        results['metadatas']
    )):
        print(f"\n[{i+1}] {doc_id}")
        print(f"    Type: {metadata['chunk_type']}")
        print(f"    Size: {len(doc)} chars")
        print(f"    Content preview:")
        print(f"    {doc[:200]}...")

inspect_function_chunks("curve_fit")

---

## Module 2 Summary

### What We Built

1. **Scraper**: Can fetch live SciPy documentation (or use sample data)
2. **Chunker**: Multiple strategies with code-awareness
3. **Vector Store**: Persistent ChromaDB with metadata

### Key Takeaways

- **Scraping**: Be polite (rate limiting, user-agent, caching)
- **Chunking**: Use code-aware strategies for documentation
- **Metadata**: Essential for filtering and debugging
- **Testing**: Always validate retrieval quality

### Next Module

In **Module 3**, we'll build the complete RAG pipeline:
- Retrieval logic with preprocessing
- Prompt engineering for code generation
- Integration with OpenAI and Ollama

---

## Exercises

1. **Add more modules**: Scrape additional SciPy modules and add to the vector store
2. **Experiment with chunk sizes**: Try 400, 800, 1200 char chunks and compare retrieval
3. **Semantic chunking**: Implement semantic chunking using the embedding function
4. **Add more metadata**: Include difficulty level, related functions, etc.